In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import pyarrow as pa
import pyarrow.parquet as pq
import time
from fake_useragent import UserAgent
import random
import json
from pprint import pprint
import csv
import os

In [6]:
session=requests.Session()
url='https://irr.by/cars/passenger'
ua=UserAgent()
links_auto=[]
CSV_FILE='auto_data.csv'

In [ ]:
def get_links(url):
    page=1
    while True:
        headers={'User-Agent':ua.random}
        params={'page':page}

        response=session.get(url, headers=headers, params=params)

        if response.status_code!=200:
            break

        if 'Извините, но объявлений по Вашему запросу не найдено.' in response.text:
            print(f'страница {page-1} была последняя')
            break

        soup=BeautifulSoup(response.text, "html.parser")

        div_cards=soup.find_all('div', class_=re.compile('ListingWrapperPage-module__'))

        if not div_cards:
            print('карточки авто не найдены')
            break

        for div in div_cards:
            clink=div.find('a')

            if clink and clink.has_attr('href'):
                href=clink['href']
                if href.startswith('/'):
                    continue
                links_auto.append(href)

        sleep=random.uniform(1, 4)
        print(f'sleep {sleep:.2f}s')
        time.sleep(sleep)
        page+=1

get_links(url)
print(links_auto)

sleep 3.00s
sleep 1.71s
sleep 2.61s
sleep 2.81s
sleep 2.68s
sleep 3.33s
sleep 2.89s
sleep 3.74s
sleep 1.33s
sleep 3.03s
sleep 2.73s
sleep 2.24s
sleep 3.52s
sleep 1.76s
sleep 3.51s
sleep 1.62s
sleep 2.04s
sleep 3.22s
sleep 1.96s
sleep 2.97s
sleep 2.58s
sleep 1.06s
sleep 3.31s
sleep 1.43s
sleep 1.92s
sleep 2.43s
sleep 1.78s
sleep 3.42s
sleep 1.08s
sleep 3.89s
sleep 3.45s
sleep 3.48s
sleep 1.48s
sleep 3.78s
sleep 1.97s
sleep 1.82s
sleep 1.97s
sleep 3.83s
sleep 3.70s
sleep 3.27s
sleep 3.24s
sleep 3.83s
sleep 2.45s
sleep 2.57s
sleep 1.58s
sleep 1.29s
sleep 2.67s
sleep 3.36s
sleep 3.64s
sleep 2.58s
sleep 2.04s
sleep 3.54s
sleep 2.65s
sleep 2.22s
sleep 3.86s
sleep 1.94s
sleep 1.94s
sleep 3.55s
sleep 2.36s
sleep 1.58s
sleep 3.78s
sleep 1.85s
sleep 1.31s
sleep 1.65s
sleep 3.22s
sleep 3.96s
sleep 3.40s
sleep 1.18s
sleep 2.11s
sleep 3.91s
sleep 1.22s
sleep 3.74s
sleep 3.06s
sleep 3.38s
sleep 1.80s
sleep 3.74s
sleep 3.41s
sleep 1.86s
sleep 2.50s
sleep 1.82s
sleep 3.71s
sleep 3.41s
sleep 1.66s
slee

ReadTimeout: HTTPSConnectionPool(host='irr.by', port=443): Read timed out. (read timeout=None)

In [9]:
print(len(links_auto))
print(links_auto)

4641
['https://irr.by/ads/advert16066909', 'https://irr.by/ads/advert16066885', 'https://irr.by/ads/advert16066559', 'https://irr.by/ads/advert16066557', 'https://irr.by/ads/advert16066531', 'https://irr.by/ads/advert16066292', 'https://irr.by/ads/advert16066255', 'https://irr.by/ads/advert16065885', 'https://irr.by/ads/advert16065742', 'https://irr.by/ads/advert16042770', 'https://irr.by/ads/advert16061647', 'https://irr.by/ads/advert16064876', 'https://irr.by/ads/advert16064751', 'https://irr.by/ads/advert16064714', 'https://irr.by/ads/advert16064587', 'https://irr.by/ads/advert16064556', 'https://irr.by/ads/advert16064312', 'https://irr.by/ads/advert16063204', 'https://irr.by/ads/advert16063189', 'https://irr.by/ads/advert16063015', 'https://irr.by/ads/advert16062489', 'https://irr.by/ads/advert16061644', 'https://irr.by/ads/advert16061477', 'https://irr.by/ads/advert16061074', 'https://irr.by/ads/advert16060928', 'https://irr.by/ads/advert16058770', 'https://irr.by/ads/advert160582

In [6]:
with open('links_auto.txt', 'w') as f:
    for link in links_auto:
        f.write(link+'\n')

In [8]:
with open('links_auto.txt', 'r') as f:
    links_auto=f.read().splitlines()

In [ ]:
def safe_int(value, default=0):
    try:
        return int(value) if value is not None else default
    except(ValueError, TypeError):
        return default

def safe_float(value, default=0):
    try:
        return float(value) if value is not None else default
    except(ValueError, TypeError):
        return default

def get_data(url):
    headers = {'User-Agent': ua.random}

    max_retries=3
    retry_delay=60
    for i in range(1, max_retries+1):
        try:
            response=session.get(url, headers=headers, timeout=15)
            break
        except requests.exceptions.RequestException as e:
            print(f"сбой доступа, попытка {i}/{max_retries} не удалась, ошибка {e}")
            if i<max_retries:
                print("ожидаем минуту")
                time.sleep(retry_delay)
            else:
                print(f'{max_retries} попытки исчерпаны, пропускаем ссылку {url}')
                return None
    
    
    if response.status_code!=200:
        return None
    
    soup=BeautifulSoup(response.text, 'html.parser')
    
    json_data=soup.find('script', type='application/ld+json')
    
    if json_data:
        data=json.loads(json_data.string)
        
        car_info=data[0] if isinstance(data, list) else data
        
        title=car_info.get('name')
        price=car_info.get('offers', {}).get('price')
        currency=car_info.get('offers', {}).get('priceCurrency')
        description=car_info.get('description')
        
        specs={}
        
        for prop in car_info.get('additionalProperty', []):
            specs[prop['name']]= prop['value']
            
        car_data = {
            'name': title,
            'price': safe_float(price),
            'currency': currency,
            'brand': specs.get('Марка'),
            'model': specs.get('Модель'),
            'year': safe_int(specs.get('Год выпуска')),
            'mileage': safe_float(specs.get('Пробег')),
            'body_type': specs.get('Тип кузова'),
            'engine_volume': safe_float(specs.get('Объем двигателя')),
            'fuel_type': specs.get('Тип двигателя'),
            'transmission': specs.get('Тип трансмиссии'),
            'color': specs.get('Цвет'),
            'description': description,
            'link': url
        }
        return car_data
    else:
        print('no json')
        return None

In [ ]:
FIELDS=['name', 'price', 'currency', 'brand', 'model', 'year', 'mileage', 'body_type', 'engine_volume', 'fuel_type', 'transmission', 'color', 'description', 'link']

def save_csv(car_data, filename=CSV_FILE):
    if car_data is None:
        return
    filen=os.path.isfile(filename)

    with open(filename, 'a', newline='', encoding='utf-8-sig') as f:
        writer=csv.DictWriter(f, fieldnames=FIELDS, delimiter=';')

        if not filen:
            writer.writeheader()

        writer.writerow(car_data)

In [13]:
for link in links_auto:
    save_csv(get_data(link))
    sleep=random.uniform(2, 5)
    print(f'sleep {sleep:.2f}s')
    time.sleep(sleep)

sleep 4.65s
sleep 3.46s
sleep 4.97s
sleep 4.34s
sleep 2.68s
sleep 2.01s
sleep 2.44s
sleep 3.04s
sleep 3.36s
sleep 3.94s
sleep 2.78s
sleep 2.90s
sleep 2.60s
sleep 4.17s
sleep 4.02s
sleep 3.98s
sleep 2.78s
sleep 3.35s
sleep 4.72s
sleep 2.81s
sleep 3.86s
sleep 3.04s
sleep 2.68s
sleep 4.10s
sleep 2.78s
sleep 3.13s
sleep 3.63s
sleep 4.06s
sleep 3.12s
sleep 4.73s
sleep 2.68s
sleep 3.48s
sleep 3.73s
sleep 3.03s
sleep 4.94s
sleep 3.45s
sleep 3.77s
sleep 2.50s
sleep 2.35s
sleep 4.57s
sleep 4.25s
sleep 3.69s
sleep 3.20s
sleep 3.97s
sleep 2.28s
sleep 3.66s
sleep 2.86s
sleep 2.33s
sleep 2.09s
sleep 4.22s
sleep 3.48s
sleep 2.76s
sleep 4.55s
sleep 2.26s
sleep 3.27s
sleep 3.16s
sleep 2.71s
sleep 3.34s
sleep 3.70s
sleep 3.48s
sleep 3.40s
sleep 2.53s
sleep 2.04s
sleep 3.33s
sleep 3.16s
sleep 2.91s
sleep 3.67s
sleep 4.88s
sleep 4.95s
sleep 3.09s
sleep 3.43s
sleep 3.78s
sleep 4.92s
sleep 3.81s
sleep 4.83s
sleep 3.15s
sleep 4.62s
sleep 4.88s
sleep 3.47s
sleep 3.24s
sleep 3.15s
sleep 4.46s
sleep 2.62s
slee

In [15]:
df=pd.read_csv('auto_data.csv', sep=';', encoding='utf-8-sig')

if 'description' in df.columns:
    df['description']=df['description'].astype(str).str.replace(r'[\r\n]+', ' ', regex=True)

df.to_csv('auto_data.csv', sep=';', index=False, encoding='utf-8-sig')

In [ ]:
df=pd.read_csv('auto_data.csv', sep=';', encoding='utf-8-sig')
schema=pa.schema([
    ('name', pa.string()),
    ('price', pa.int64()),
    ('currency', pa.string()),
    ('brand', pa.string()),
    ('model', pa.string()),
    ('year', pa.int64()),
    ('mileage', pa.int64()),
    ('body_type', pa.string()),
    ('engine_volume', pa.float64()),
    ('fuel_type', pa.string()),
    ('transmission', pa.string()),
    ('color', pa.string()),
    ('description', pa.string()),
    ('link', pa.string())
])

table=pa.Table.from_pandas(df, schema=schema)
pq.write_table(table, 'auto.parquet')

In [ ]:
tabler = pq.read_table("auto.parquet")
dfr = tabler.to_pandas()

# --- Output example ---

# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_colwidth', None)

# display(dfr)

# ----------------------

,name,price,currency,brand,model,year,mileage,body_type,engine_volume,fuel_type,transmission,color,description,link
0,По всем вопросам звонить - Объявление 16066909,26741,BYN,KIA,Rio,2013,147000,седан,1.6,бензин,механика,белый металлик,По всем вопросам звонить.,https://irr.by/ads/advert16066909
1,"Автомобиль в отличном состоянии, крепкий кузов без жуков и ржавчины, никаких проблем, ошиб - Объявление 16066885",8864,BYN,Opel,Astra,2001,295000,универсал,1.7,дизель,механика,зеленый металлик,"Автомобиль в отличном состоянии, крепкий кузов без жуков и ржавчины, никаких проблем, ошибок и неисправностей по автомобилю нет все работает, ручник держит, подвеска целая не стучи",https://irr.by/ads/advert16066885
2,"Стильный, экономичный и надежный городской хэтчбек :Ищете надежный, маневренный и очень эк - Объявление 16066559",14927,BYN,Ford,Ka,2009,217000,хэтчбек,1.2,бензин,механика,NaN,"Стильный, экономичный и надежный городской хэтчбек :Ищете надежный, маневренный и очень экономичный автомобиль для ежедневных поездок? Этот Ford Ka второго поколения — идеальный вы",https://irr.by/ads/advert16066559
3,"Стильный, экономичный и надежный городской хэтчбек :Ищете надежный, маневренный и очень эк - Объявление 16066557",14927,BYN,Ford,Ka,2009,217000,хэтчбек,1.2,бензин,механика,NaN,"Стильный, экономичный и надежный городской хэтчбек :Ищете надежный, маневренный и очень экономичный автомобиль для ежедневных поездок? Этот Ford Ka второго поколения — идеальный вы",https://irr.by/ads/advert16066557
4,В продаже renault sandero 2017 года выпуска.С момента покупки в салоне машина 9 лет в одни - Объявление 16066531,23000,BYN,Renault,Sandero,2017,182000,хэтчбек,1.6,бензин,механика,белый,"В продаже renault sandero 2017 года выпуска.С момента покупки в салоне машина 9 лет в одних руках.Техническое и внешнее состояние машины не требует вложений, все агрегаты обслужены",https://irr.by/ads/advert16066531
5,"Новая летняя резина и пара зимней. Новый АКБ. Чистый и ухоженный салон, внутри не курили. - Объявление 16066292",17258,BYN,Peugeot,3008,2011,244000,NaN,0.0,бензин,автомат,белый,"Новая летняя резина и пара зимней. Новый АКБ. Чистый и ухоженный салон, внутри не курили. Снят с учёта. Имеются возрастные нюансы по кузову, так же нужна переварка левого порога. Т",https://irr.by/ads/advert16066292
6,"Этот вариант подойдет для машины в хорошем состоянии, которую жалко продавать. Заголовок: - Объявление 16066255",1,BYN,Audi,80,1992,452632,универсал,2.0,бензин,механика,зеленый,"Этот вариант подойдет для машины в хорошем состоянии, которую жалко продавать. Заголовок: Audi 80 — Классика, которая не стареет. Это не просто автомобиль, это характер. Audi 80 —",https://irr.by/ads/advert16066255
7,"Hyundai Accent, седан, 2018 г.в., автомат, 1.6 л, цвет: черный - Объявление 16065885",39362,BYN,Hyundai,Accent,2018,89000,седан,1.6,бензин,автомат,черный,"Надежный автомобиль в ухоженном состоянии. За все время 2 владельца. К машине относились бережно, в салоне не курили. Машина своевременно обслуживалась, на руках есть все чеки, зак",https://irr.by/ads/advert16065885
8,"Hyundai i30, универсал, 2018 г.в., механика, 1.4 л, цвет: коричневый металлик - Объявление 16065742",34700,BYN,Hyundai,i30,2018,212000,универсал,1.4,бензин,механика,коричневый металлик,"Купил в салоне с пробегом 145000, машина из РП. Аварий не было. Техническое состояние хорошее, есть незначительные дефекты по корпусу",https://irr.by/ads/advert16065742
9,"BMW 118, хэтчбек, 2006 г.в., автомат, 2 л, цвет: синий металлик - Объявление 16042770",19979,BYN,BMW,118,2006,305000,хэтчбек,2.0,бензин,автомат,синий металлик,"Надёжный, комфортный, скоростной, функциональный, стильный автомобиль BMW118i, кузов E87, с в меру мягкой подвеской и отличной управляемостью. В автомобиле есть андроид магнитола с",https://irr.by/ads/advert16042770
